In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option('display.max_columns',1000)
pd.set_option('display.max_rows',1000)

import lxml
import html5lib
from urllib.request import urlopen
import time

from bs4 import BeautifulSoup
import requests

In [2]:
## Given the url that refers to a specific pitcher and season
## we scrape the data and process it a bit
def get_season_pitching_data(url):    
    time.sleep(1)
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    html=list(soup.children)[-1]
    body = list(html.children)[-1]
    sec_next = list(body.children)
    secnum = np.where(["Opponent" in str(x) for x in sec_next])[0][0]
    key_section = sec_next[secnum]
    working_part = list(key_section.children)
    p_header = working_part[0].strip().split()
    mod_header= ['at_vs','Opponent','League', 'GS', 'CG', 'SHO', 'GF', 'SV', 'IP', 'H',
            'BFP', 'HR', 'R', 'ER', 'BB', 'IB', 'SO', 'SH', 'SF', 'WP', 'HBP',
            'BK', 'x2B', 'x3B', 'GDP', 'ROE', 'W', 'L', 'ERA']

    date_list = []
    day_href_list = []
    for k in range(1,len(working_part),4):
        date_list.append(working_part[k].get_text().strip())
        day_href_list.append(working_part[k].attrs['href'])

    dblhead_num_list = []
    for k in range(2,len(working_part),4):
        dblhead_num_list.append(working_part[k].strip())

    game_href_list = []
    for k in range(3,len(working_part),4):
        game_href_list.append(working_part[k].attrs['href'])

    main_data_matrix = []
    for k in range(4,len(working_part),4):
        main_data_row = (working_part[k].strip().split())[:29]
        main_data_matrix.append(main_data_row)

    out_df = pd.DataFrame(main_data_matrix, columns = mod_header)
    out_df['date'] = date_list
    out_df['dblhead_num'] = dblhead_num_list
    return out_df

In [3]:
### Get the links to the pitcher-season tables given the pitcher id
def get_daily_season_links(pitcher_id):
    letter = pitcher_id.upper()[0]
    url_prefix = 'https://www.retrosheet.org/boxesetc/'
    url = url_prefix+letter+'/P'+pitcher_id+'.htm'
    time.sleep(1)
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    html=list(soup.children)
    body = list(html[2].children)[5]
    pre_texts = [x for x in body.find_all('pre')]
    secnum = np.where([x.get_text().strip().startswith('Pitching Record') for x in pre_texts])[0][0]
    a_pre_texts = pre_texts[secnum].find_all('a')
    daily_season_links = [url_prefix+x.attrs['href'][3:] for x in a_pre_texts if x.get_text()=='Daily']
    return daily_season_links

In [4]:
# Get all the data for a particular pitcher
def get_full_pitching_data(pitcher_id):
    link_list = get_daily_season_links(pitcher_id)
    df_pitching = pd.DataFrame()
    for url in link_list:
        df_pitching = pd.concat((df_pitching, get_season_pitching_data(url)))
    return df_pitching

In [5]:
df = pd.read_csv('data/df_mlb3.csv')

/var/folders/y0/_yc3t8mn3kx8j1td1w609rl00000gn/T/ipykernel_77611/1258349196.py:1: DtypeWarning: Columns (72,73,185) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/df_mlb3.csv')


In [6]:
start_pitchers_h = df.starting_pitcher_id_h.unique()
start_pitchers_v = df.starting_pitcher_id_v.unique()
len(start_pitchers_h), len(start_pitchers_v)

(2874, 2903)

In [9]:
start_pitchers_all = np.union1d(start_pitchers_h, start_pitchers_v)
len(start_pitchers_all), start_pitchers_all[:10]

np.where(start_pitchers_all == 'mesaj001') # 1841

(array([1841]),)

In [12]:
# run this for everyone in the list - may take a bit to run...
## better to filter based on year, i.e., no need in storing data prior to 1980 season..
for (idx, p_id) in enumerate(start_pitchers_all[1841:]):
    print(f'{idx}: {p_id}')
    df_temp = get_full_pitching_data(p_id)
    # may want to modify below to save to a dedicated folder
    fname_out = 'data/pitch/pitching_data_'+p_id+'.csv'
    df_temp.to_csv(fname_out, index=False)

0: mesaj001
1: meyea001
2: meyed002
3: meyem001
4: miced001
5: michc001
6: middj001
7: middk001
8: mikom001
9: milab001
10: milew001
11: milis001
12: milla001
13: milla002
14: milla003
15: millb001
16: millb004
17: millb005
18: millj002
19: millk003
20: millk004
21: millm004
22: millp001
23: mills001
24: millt001
25: millt002
26: millt003
27: millw001
28: milot001
29: milte001
30: mimbm001
31: mincn001
32: minec101
33: minez001
34: minom001
35: mintg001
36: minug001
37: miraa001
38: miraa002
39: mirap001
40: miscp001
41: mitcb001
42: mitcj001
43: mitcp101
44: mitrs001
45: mizec001
46: mlicd001
47: mlodc001
48: mmahk001
49: mockg001
50: moehb001
51: moeld001
52: mohlm001
53: molls001
54: monac001
55: montf001
56: montj001
57: montj002
58: montj004
59: montm002
60: montr004
61: moode001
62: moonb001
63: moora002
64: moorb101
65: moorm001
66: moorm003
67: moort001
68: moraf001
69: morea001
70: morea101
71: mored002
72: morga001
73: morge001
74: morgm001
75: morij001
76: morrb001
77: morrc

In [13]:
np.where(start_pitchers_all == 'zycht001')

(array([3120]),)

In [15]:
len(start_pitchers_all)

3121